# Quick start

A two-mode switched system is simulated, then learned from its trajectories alone (passive learning), and the learned model is saved, reloaded and simulated.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import hydyreg as hd

## A system to learn

A relay oscillator: $\dot x_1 = x_2$, $\dot x_2 = -x_1 - 0.2\,x_2 + u$ with $u = -1$ when $x_1 > 0$ (mode 1) and $u = +1$ otherwise (mode 2). `SNDS_affine` takes, for each mode, the rows $[a_{d1}, a_{d2}, b_d]$ of $\dot x_d = a_{d1} x_1 + a_{d2} x_2 + b_d$, the switching surfaces and a function giving the mode of a state.

In [ ]:
params = np.array([[[0, 1, 0], [-1, -0.2, -1]],     # mode 1 (x1 > 0): u = -1
                   [[0, 1, 0], [-1, -0.2, +1]]])    # mode 2 (x1 <= 0): u = +1
surface = lambda t, x: x[0]
surface.terminal, surface.direction = True, 0
system = hd.SNDS_affine(params, [surface], lambda x: 0 if x[0] > 0 else 1)

data = system.simulate_trajectory(0.02, 12, [[2.5, 0.0], [-1.5, 1.5], [0.5, -2.0]])
hd.plotting.plot_trajectories(data, labelfun=system.labelfun);

## Learning from the trajectories

The incremental method finds the number of modes. `seg_tol` is the size of a derivative jump that marks a switch (here $\dot x_2$ jumps by 2).

In [ ]:
config = hd.LearnConfig(method="incremental", poly_degree=1, boundary_degree=1, seg_tol=0.1)
result = hd.learn_from_data(data, config)
print(result.summary())
print(result.equations())

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
result.plot("regions", ax=axs[0], true_labelfun=system.labelfun)
result.plot("comparison", ax=axs[1]);

The same equations in LaTeX, ready to paste into a document:

In [ ]:
print(result.equations("latex"))

## Saving, loading and simulating the model

In [ ]:
hd.save_model(result.model, "relay_model.json")
model = hd.load_model("relay_model.json")
sim = model.to_SNDS().simulate_trajectory(0.02, 12, [[1.0, 1.0]])
ref = system.simulate_trajectory(0.02, 12, [[1.0, 1.0]])
print("largest difference along the trajectory:", np.abs(sim.x_list[0] - ref.x_list[0]).max())